## Setup

In [54]:
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO_ROOT = Path.cwd().resolve()
print(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.append("..")

/Users/tzhang04/Desktop/active-passive-alternations/notebooks


In [57]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import normaltest, ttest_rel, wilcoxon
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display
from src.uid import *

/Users/tzhang04/Desktop/active-passive-alternations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-01 13:02:22 INFO: Downloaded file to /Users/tzhang04/stanza_resources/resources.json
2026-04-01 13:02:22 INFO: Downloading default packages for language: en (English) ...
2026-04-01 13:02:23 INFO: File exists: /Users/tzhang04/stanza_resources/en/default.zip
2026-04-01 13:02:24 INFO: Finished downloading models and saved to /Users/tzhang04/stanza_resources
2026-04-01 13:02:24 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-01 13:02:25 INFO: Downloaded file to /Users/tzhang04/stanza_resources/resources.json
2026-04-01 13:02:25 INFO: Loading t

In [58]:
import conllu
from conllu import Token, parse, parse_tree
from pyinflect import getAllInflections, getInflection
from src.utils import *
from src.units.word import *
from src.units.sentence import *

## Test Feature Extraction

In [4]:
ud_path = "../data/en_gum-ud-dev.conllu"
docs = iter_counterfactual_docs(ud_path)

In [5]:
docs[0]

('f::GUM_academic_exposure::-1::og',
 ['Introduction',
  'Research on adult-learned second language (L2) has provided considerable insight into the neurocognitive mechanisms underlying the learning and processing of L2 grammar [1]–[11].',
  'Of interest here, studies suggest that, despite the difficulties in acquiring L2 grammar, adult learners can approximate native-like levels of use and neurocognitive processing [12]–[15].',
  'However, it is not enough to have attained such native-like levels.',
  'Crucially, it is also desirable to retain them, even in the absence of continued practice or exposure to the L2.',
  'In fact, substantial periods (months to years) of limited or no exposure following L2 training are not uncommon, and may even be the norm [16].',
  'Such a scenario may be found in different situations, including when one studies a language in a classroom and then stops taking classes [17], [18] and when one is immersed in a foreign language setting and then moves away [1

In [6]:
doc_id, sents, doc = docs[0]

In [7]:
for doc_id, sents, doc in docs:
    for sent in doc:
        try:
            active_sent = ActiveSentence(sent)
        except:
            continue

In [8]:
active_sent.text

'If you wash your overalls alone or in a light load, use about half the detergent called for and less water.'

In [9]:
active_sent.active_subject_word

{'id': 2,
 'form': 'you',
 'lemma': 'you',
 'upos': 'PRON',
 'xpos': 'PRP',
 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '2', 'PronType': 'Prs'},
 'head': 3,
 'deprel': 'nsubj',
 'deps': [('nsubj', 3)],
 'misc': {'Entity': '(3-person-giv:inact-nnsnn-cf1-1-ana)'},
 'inflection': 'PRP',
 'children': []}

In [10]:
tok, model, device = load_lm('distilgpt2', device='mps')
unigram = UnigramLM(tok)
unigram.fit("\n".join(["\n".join(doc[1]) for doc in docs]), uid_unit='word')

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2572.36it/s, Materializing param=transformer.wte.weight]            
GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (487324 > 1024). Running this sequence through the model will result in indexing errors


In [11]:
result = ling_features(active_sent, tok, unigram, uid_unit='word')

In [12]:
unigram.model.get('you', 0)

0.01065558639014376

In [13]:
unigram('you')

(array([0.01065559]), np.float64(0.01065558639014376))

In [15]:
result

{'agent': [{'id': 2,
   'form': 'you',
   'lemma': 'you',
   'upos': 'PRON',
   'xpos': 'PRP',
   'feats': {'Case': 'Nom',
    'Number': 'Sing',
    'Person': '2',
    'PronType': 'Prs'},
   'head': 3,
   'deprel': 'nsubj',
   'deps': [('nsubj', 3)],
   'misc': {'Entity': '(3-person-giv:inact-nnsnn-cf1-1-ana)'},
   'inflection': 'PRP',
   'children': []}],
 'agent_len': 1,
 'agent_unigram_prob': np.float64(0.01065558639014376),
 'agent_is_pronoun': True,
 'agent_is_plural': False,
 'patient': [{'id': 4,
   'form': 'your',
   'lemma': 'your',
   'upos': 'PRON',
   'xpos': 'PRP$',
   'feats': {'Case': 'Gen',
    'Number': 'Sing',
    'Person': '2',
    'Poss': 'Yes',
    'PronType': 'Prs'},
   'head': 5,
   'deprel': 'nmod:poss',
   'deps': [('nmod:poss', 5)],
   'misc': {'Entity': '(1-object-giv:inact-sssss-cf2-2-coref(3-person-giv:act-nnsnn-cf1-1-ana)'},
   'inflection': 'PRP$',
   'children': []},
  {'id': 5,
   'form': 'overalls',
   'lemma': 'overall',
   'upos': 'NOUN',
   'xpos': 

In [5]:
# Animacy from WordNet
import nltk
nltk.download("wordnet")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/tzhang04/nltk_data...


In [91]:
syn = wn.synsets("we", pos=wn.NOUN)
i = 0
syn[i], syn[i].lexname(), syn[i].definition()

IndexError: list index out of range

In [92]:
syn

[]

In [90]:
is_animate("we")

False

#### Check outputs

In [79]:
out = pd.read_csv("../temp/cf_word_sentence_uid_chkpt.csv")

In [82]:
ling_features = []
for name in ['agent', 'patient']:
    ling_features.extend([
        f"{name}",
        f"{name}_len", 
        f"{name}_unigram_prob",
        f"{name}_is_pronoun",
        f"{name}_is_plural",
        f"{name}_is_animate",])

In [88]:
ling_feat_out = out[['sentence'] + ling_features]
ling_feat_out.iloc[25]

sentence                (Note that we do not consider case studies, pu...
agent                                                                  we
agent_len                                                               1
agent_unigram_prob                                               0.004854
agent_is_pronoun                                                     True
agent_is_plural                                                      True
agent_is_animate                                                    False
patient                 case studies, purely observational data, or re...
patient_len                                                            12
patient_unigram_prob                                                  0.0
patient_is_pronoun                                                  False
patient_is_plural                                                    True
patient_is_animate                                                  False
Name: 25, dtype: object